In [ ]:
import os
import requests
from getpass import getpass
import pandas as pd

SERVER_URL = "https://kf.kobotoolbox.org"
API_TOKEN = (os.getenv("KOBO_API_TOKEN") or getpass("Paste Kobo API token: ")).strip()

headers = {
    "Authorization": f"Token {API_TOKEN}",
    "Accept": "application/json",
}

# 1. Fetch the LIST of all assets instead of a specific UID
list_url = f"{SERVER_URL}/api/v2/assets/"
r = requests.get(list_url, headers=headers)
r.raise_for_status()
assets = r.json().get('results', [])

# 2. Find "GDSC_Authoring" in that list
target_asset = None
for asset in assets:
    if asset['name'].lower() == "gdsc_authoring":
        target_asset = asset
        print(f"Success! Found '{asset['name']}' with UID: {asset['uid']}")
        break

if target_asset:
    # 3. Get the data for this specific form
    data_url = target_asset.get("data")
    data_response = requests.get(data_url, headers=headers)
    data_response.raise_for_status()
    
    # 4. Convert to DataFrame
    df = pd.DataFrame(data_response.json()['results'])
    print(f"Total submissions retrieved: {len(df)}")
    display(df.head())
else:
    print("Could not find a form named 'GDSC_Authoring'. Check your spelling!")
